# ⚡ TensorFlow — Complete Revision Guide

> **TensorFlow 2.x** by Google. Uses **eager execution** by default (like NumPy).  
> `tf.keras` is the official high-level API. `@tf.function` compiles code to a fast graph.

---

## 📚 What You Will Learn
1. Setup & TF Ecosystem  
2. Tensors (immutable) vs Variables (mutable)  
3. GradientTape — automatic differentiation  
4. Building models with tf.keras  
5. Custom training loop  
6. tf.data pipeline (fast data loading)  
7. Callbacks & TensorBoard  
8. CNN & LSTM  
9. Transfer Learning  
10. tf.function — graph mode  
11. Saving models for production  
12. Interview Q&A  

---
## 1️⃣ Installation & Setup

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses, optimizers, metrics, callbacks
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')
print(f'Eager execution    : {tf.executing_eagerly()}')

# List available devices
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')
print(f'CPUs: {[d.name for d in cpus]}')
print(f'GPUs: {gpus if gpus else "None — running on CPU"}')

# Optional: prevent GPU memory hoarding (use only what's needed)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

---
## 2️⃣ Tensors vs Variables

| | `tf.constant` | `tf.Variable` |
|---|---|---|
| Mutable? | ❌ No | ✅ Yes |
| Use case | Data, intermediate values | Model weights |
| Tracked by tape? | Only if watched | Always |
| Method to update | Create new tensor | `.assign()` |

In [ ]:
import tensorflow as tf
import numpy as np

print('=== tf.constant — IMMUTABLE ===')  
c1 = tf.constant([1, 2, 3], dtype=tf.float32)
c2 = tf.constant([[1.0, 2.0], [3.0, 4.0]])
c3 = tf.zeros((3, 4))
c4 = tf.ones((2, 3))
c5 = tf.random.normal((3, 3), mean=0, stddev=1)
c6 = tf.random.uniform((3, 3), minval=0, maxval=1)
c7 = tf.eye(3)
c8 = tf.range(0, 10, 2)

print(f'1-D: {c1.numpy()}')
print(f'2-D:\n{c2.numpy()}')
print(f'range(0,10,2): {c8.numpy()}')
print(f'shape: {c2.shape}  dtype: {c2.dtype}  ndim: {c2.ndim}')

print('\n=== Tensor Operations ===')
a = tf.constant([[1., 2.], [3., 4.]])
b = tf.constant([[5., 6.], [7., 8.]])

print(f'a + b:\n{tf.add(a, b).numpy()}')
print(f'a * b (element-wise):\n{tf.multiply(a, b).numpy()}')
print(f'a @ b (matmul):\n{tf.matmul(a, b).numpy()}')
print(f'reduce_sum(a): {tf.reduce_sum(a).numpy()}')
print(f'reduce_mean axis=0: {tf.reduce_mean(a, axis=0).numpy()}')

print('\n=== Reshaping ===')
t = tf.random.normal((2, 3, 4))
print(f'Original        : {t.shape}')
print(f'reshape(6, 4)   : {tf.reshape(t, (6, 4)).shape}')
print(f'expand_dims(0)  : {tf.expand_dims(t, 0).shape}')
print(f'squeeze         : {tf.squeeze(tf.zeros((1,3,1,4))).shape}')
print(f'transpose       : {tf.transpose(a).shape}')
print(f'concat axis=0   : {tf.concat([a, b], axis=0).shape}')

print('\n=== NumPy interop ===')
arr = np.array([1.0, 2.0, 3.0])
t   = tf.constant(arr)          # NumPy → TF
back = t.numpy()                # TF → NumPy
print(f'type(arr)    : {type(arr)}')
print(f'type(t)      : {type(t)}')
print(f'type(t.numpy): {type(back)}')

In [ ]:
import tensorflow as tf

print('=== tf.Variable — MUTABLE ===')
w = tf.Variable(tf.random.normal((3, 3)), name='weights')
b = tf.Variable(tf.zeros((3,)),           name='bias')

print(f'w.name     : {w.name}')
print(f'w.shape    : {w.shape}')
print(f'w.trainable: {w.trainable}')
print(f'w initial value:\n{w.numpy().round(3)}')

# Modify in place
w.assign(tf.zeros((3, 3)))               # Replace entire value
print(f'\nAfter assign(zeros):\n{w.numpy()}')

w.assign_add(tf.ones((3, 3)) * 2)        # Add 2
print(f'\nAfter assign_add(2):\n{w.numpy()}')

w.assign_sub(tf.ones((3, 3)))            # Subtract 1
print(f'\nAfter assign_sub(1):\n{w.numpy()}')

---
## 3️⃣ GradientTape — TensorFlow's Autograd

TensorFlow records operations in a **tape**. When you call `tape.gradient()`, it plays the tape backwards to compute derivatives.

> Equivalent to PyTorch's `loss.backward()` + `param.grad`.

In [ ]:
import tensorflow as tf

# ── Basic gradient ────────────────────────────────────────
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x**2 + 2*x + 1       # y = x² + 2x + 1

dy_dx = tape.gradient(y, x)  # dy/dx = 2x + 2
print(f'x = {x.numpy()}')
print(f'y = x² + 2x + 1 = {y.numpy()}')
print(f'dy/dx = 2x + 2 = {dy_dx.numpy()}')   # 2*3+2 = 8.0

print('\n── Multiple variables ──')
w = tf.Variable(2.0)
b = tf.Variable(1.0)

with tf.GradientTape() as tape:
    y = w * 3.0 + b             # y = 2*3 + 1 = 7

grads = tape.gradient(y, [w, b])
print(f'y = w*x + b = {y.numpy()}')
print(f'dy/dw = x = {grads[0].numpy()}')  # 3.0
print(f'dy/db = 1 = {grads[1].numpy()}')  # 1.0

print('\n── Watching a non-Variable tensor ──')
x_const = tf.constant(5.0)       # Constants not watched by default

with tf.GradientTape() as tape:
    tape.watch(x_const)           # Explicitly watch it
    z = x_const ** 3

dz_dx = tape.gradient(z, x_const)
print(f'dz/dx = 3x² = {dz_dx.numpy()}')   # 3 * 25 = 75

print('\n── Persistent tape (multiple gradients) ──')
x2 = tf.Variable(4.0)

with tf.GradientTape(persistent=True) as tape:
    y1 = x2 ** 2    # 16
    y2 = x2 ** 3    # 64

print(f'dy1/dx = 2x = {tape.gradient(y1, x2).numpy()}')   # 8.0
print(f'dy2/dx = 3x²= {tape.gradient(y2, x2).numpy()}')   # 48.0
del tape   # Free resources when persistent=True

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ── Manual linear regression using GradientTape ──────────
# y = 3x + 2 (true relationship)
np.random.seed(42)
X_lr = np.linspace(-1, 1, 100).astype(np.float32)
y_lr = 3.0 * X_lr + 2.0 + np.random.normal(0, 0.2, 100).astype(np.float32)

# Learnable parameters
W = tf.Variable(0.0, name='weight')
b = tf.Variable(0.0, name='bias')

lr = 0.1
losses_lr = []

for step in range(200):
    with tf.GradientTape() as tape:
        y_pred = W * X_lr + b
        loss   = tf.reduce_mean((y_pred - y_lr) ** 2)   # MSE

    dW, db = tape.gradient(loss, [W, b])
    W.assign_sub(lr * dW)
    b.assign_sub(lr * db)
    losses_lr.append(loss.numpy())

print(f'Learned W: {W.numpy():.4f} (true: 3.0)')
print(f'Learned b: {b.numpy():.4f} (true: 2.0)')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.scatter(X_lr, y_lr, alpha=0.5, label='Data')
ax1.plot(X_lr, W.numpy()*X_lr + b.numpy(), 'r-', linewidth=2, label=f'Learned: {W.numpy():.2f}x + {b.numpy():.2f}')
ax1.set_title('Linear Regression via GradientTape'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(losses_lr); ax2.set_title('Loss over Steps'); ax2.set_xlabel('Step'); ax2.set_ylabel('MSE'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4️⃣ Building Models with tf.keras

Three ways — same as Keras (TF 2.x integrates Keras).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── Sequential ────────────────────────────────────────────
model_seq = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='sequential_mlp')
model_seq.summary()

print('\n')

# ── Functional ────────────────────────────────────────────
inputs  = keras.Input(shape=(784,), name='inputs')
x       = layers.Dense(256, activation='relu')(inputs)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.3)(x)
x       = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(10,  activation='softmax')(x)
model_func = keras.Model(inputs, outputs, name='functional_mlp')
model_func.summary()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── Subclassing ───────────────────────────────────────────
class MyMLP(keras.Model):
    def __init__(self, units=256, num_classes=10, dropout=0.3):
        super().__init__()
        self.dense1 = layers.Dense(units, activation='relu')
        self.bn1    = layers.BatchNormalization()
        self.drop1  = layers.Dropout(dropout)
        self.dense2 = layers.Dense(units // 2, activation='relu')
        self.out    = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.bn1(x, training=training)    # IMPORTANT: pass training flag
        x = self.drop1(x, training=training)  # Dropout only active during training
        x = self.dense2(x)
        return self.out(x)

model_sub = MyMLP()
_ = model_sub(tf.zeros((1, 784)))  # Build model
model_sub.summary()

---
## 5️⃣ Compile & Train on MNIST

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ── Load & preprocess MNIST ───────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

print(f'Train: {X_train.shape} → flat: {X_train_flat.shape}')
print(f'Test : {X_test.shape}  → flat: {X_test_flat.shape}')

# Show sample
fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i in range(10):
    axes[i].imshow(X_train[i], cmap='gray')
    axes[i].set_title(y_train[i])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ── Model ────────────────────────────────────────────────
model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# ── Compile ───────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',   # labels are integers
    metrics=['accuracy']
)

model.summary()

# ── Callbacks ─────────────────────────────────────────────
cb_list = [
    EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    ),
    ModelCheckpoint(
        '/tmp/best_mlp.keras', save_best_only=True,
        monitor='val_accuracy', verbose=0
    )
]

# ── Train ─────────────────────────────────────────────────
history = model.fit(
    X_train_flat, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=cb_list,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc*100:.2f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6️⃣ Custom Training Loop

Use when `model.fit()` is not flexible enough (e.g., GANs, custom loss weighting, multi-task learning).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── Build model ────────────────────────────────────────────
custom_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(10)
    # NOTE: No softmax — we use from_logits=True in loss
])

optimizer   = keras.optimizers.Adam(learning_rate=0.001)
loss_fn     = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
train_acc   = keras.metrics.SparseCategoricalAccuracy(name='train_acc')
val_acc     = keras.metrics.SparseCategoricalAccuracy(name='val_acc')

# ── tf.data pipeline ─────────────────────────────────────
BATCH_SIZE = 256

train_ds = tf.data.Dataset.from_tensor_slices((X_train_flat, y_train))
train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_X = X_test_flat[:5000]
val_y = y_test[:5000]
val_ds = tf.data.Dataset.from_tensor_slices((val_X, val_y))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ── @tf.function: compile step to graph for speed ────────
@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = custom_model(x_batch, training=True)
        loss   = loss_fn(y_batch, logits)
    gradients = tape.gradient(loss, custom_model.trainable_weights)
    optimizer.apply_gradients(zip(gradients, custom_model.trainable_weights))
    train_acc.update_state(y_batch, logits)
    return loss

@tf.function
def val_step(x_batch, y_batch):
    logits = custom_model(x_batch, training=False)
    val_acc.update_state(y_batch, logits)

# ── Training loop ─────────────────────────────────────────
for epoch in range(1, 8):
    # Reset metrics
    train_acc.reset_state()
    val_acc.reset_state()
    total_loss = 0.0
    n_batches  = 0

    # Train
    for x_b, y_b in train_ds:
        loss = train_step(x_b, y_b)
        total_loss += loss.numpy()
        n_batches  += 1

    # Validate
    for x_b, y_b in val_ds:
        val_step(x_b, y_b)

    avg_loss = total_loss / n_batches
    print(f'Epoch {epoch:2d}/7 | Loss: {avg_loss:.4f} | '
          f'Train Acc: {train_acc.result():.4f} | Val Acc: {val_acc.result():.4f}')

---
## 7️⃣ tf.data Pipeline — Efficient Data Loading

The `tf.data` API is TensorFlow's way to build **fast, scalable** input pipelines.  
It can overlap CPU data preparation with GPU training using `.prefetch()`.

In [ ]:
import tensorflow as tf
import numpy as np

# ── Basic pipeline operations ─────────────────────────────
ds = tf.data.Dataset.from_tensor_slices(
    (X_train_flat[:1000], y_train[:1000])
)

# Transformation chain
ds = (
    ds
    .shuffle(buffer_size=1000, seed=42)   # Shuffle data randomly
    .batch(32)                             # Group into batches of 32
    .prefetch(tf.data.AUTOTUNE)            # Pre-load next batch while GPU trains
)

# Inspect
for x_batch, y_batch in ds.take(2):
    print(f'Batch X: {x_batch.shape}, y: {y_batch.shape}')

print()

# ── .map() — transform each element ──────────────────────
def augment(x, y):
    # Add small Gaussian noise for augmentation
    x = x + tf.random.normal(tf.shape(x), stddev=0.05)
    x = tf.clip_by_value(x, 0.0, 1.0)
    return x, y

ds_aug = (
    tf.data.Dataset.from_tensor_slices((X_train_flat, y_train))
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)   # Apply in parallel!
    .shuffle(10000)
    .batch(128)
    .cache()                  # Cache after first epoch (avoids re-reading)
    .prefetch(tf.data.AUTOTUNE)
)

x_sample, y_sample = next(iter(ds_aug))
print(f'Augmented batch: {x_sample.shape}')
print(f'Pixel range after clip: [{x_sample.numpy().min():.3f}, {x_sample.numpy().max():.3f}]')

# ── Using dataset directly with model.fit ────────────────
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test_flat, y_test))
    .batch(256)
    .prefetch(tf.data.AUTOTUNE)
)

print('\nTraining with tf.data pipeline...')
quick_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
quick_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
quick_model.fit(ds_aug, epochs=3, validation_data=test_ds, verbose=1)

---
## 8️⃣ CNN with TensorFlow

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# ── Reshape for CNN: add channel dimension ────────────────
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test.reshape(-1, 28, 28, 1)
print(f'CNN input shape: {X_train_cnn.shape}')   # (60000, 28, 28, 1)

# ── Build CNN ─────────────────────────────────────────────
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # Block 1 — detect edges and basic patterns
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                  # 28×28 → 14×14

    # Block 2 — detect shapes (combinations of edges)
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                  # 14×14 → 7×7

    # Block 3 — detect high-level features (digit parts)
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    # Classifier head
    layers.GlobalAveragePooling2D(),             # 7×7×128 → 128
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='mnist_cnn')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)],
    verbose=1
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f'\nCNN Test Accuracy: {cnn_acc*100:.2f}%')

---
## 9️⃣ LSTM with TensorFlow

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Treat each image as 28 timesteps of 28-dim vectors
# X_train shape: (60000, 28, 28) — already the right shape for LSTM

lstm_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),  # Both directions
    layers.Dropout(0.3),
    layers.LSTM(32, return_sequences=False),   # Output last timestep only
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='mnist_lstm')

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()

lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=256,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f'\nLSTM Test Accuracy: {lstm_acc*100:.2f}%')

---
## 🔟 `@tf.function` — Graph Mode for Speed

By default TF 2.x runs eagerly (immediately). `@tf.function` **traces** your Python function once and compiles it into an optimized TensorFlow graph — significantly faster for repeated calls.

In [ ]:
import tensorflow as tf
import time

# ── Compare eager vs graph execution ─────────────────────
def heavy_computation(x):
    for _ in range(100):
        x = tf.matmul(x, x)
    return x

@tf.function
def heavy_computation_graph(x):
    for _ in range(100):
        x = tf.matmul(x, x)
    return x

x = tf.random.normal((50, 50))

# Warm-up
_ = heavy_computation(x)
_ = heavy_computation_graph(x)

# Time eager
start = time.time()
for _ in range(20):
    heavy_computation(x)
eager_time = time.time() - start

# Time graph
start = time.time()
for _ in range(20):
    heavy_computation_graph(x)
graph_time = time.time() - start

print(f'Eager execution  : {eager_time*1000:.1f}ms')
print(f'Graph (@tf.func) : {graph_time*1000:.1f}ms')
print(f'Speedup          : {eager_time/graph_time:.2f}x')

# ── Tracing behavior ─────────────────────────────────────
print('\n=== Tracing Behavior ===')

@tf.function
def multiply(x, y):
    print(f'TRACING with x={x.shape}, y={y.shape}')  # Only prints during tracing!
    return x * y

a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[2.0, 2.0], [2.0, 2.0]])

print('First call — traces:')
r = multiply(a, b)
print('Second call — reuses graph (no TRACING print):')
r = multiply(a, b)
print('Different shape — traces again:')
r = multiply(tf.ones((3,3)), tf.ones((3,3)))

---
## 1️⃣1️⃣ Saving & Loading Models

In [ ]:
import tensorflow as tf
import os

# ── Format 1: Keras native (.keras) — RECOMMENDED ────────
cnn_model.save('/tmp/cnn_model.keras')
loaded_keras = tf.keras.models.load_model('/tmp/cnn_model.keras')
_, acc_k = loaded_keras.evaluate(X_test_cnn, y_test, verbose=0)
print(f'.keras format — Loaded acc: {acc_k*100:.2f}%')

# ── Format 2: SavedModel (for TF Serving) ────────────────
cnn_model.save('/tmp/cnn_saved_model/')     # Creates a directory
loaded_sm = tf.keras.models.load_model('/tmp/cnn_saved_model/')
_, acc_sm = loaded_sm.evaluate(X_test_cnn, y_test, verbose=0)
print(f'SavedModel format — Loaded acc: {acc_sm*100:.2f}%')

# Check what's inside
print('\nSavedModel directory contents:')
for item in os.listdir('/tmp/cnn_saved_model/'):
    print(f'  {item}')

# ── Format 3: Weights only ────────────────────────────────
cnn_model.save_weights('/tmp/cnn_weights.weights.h5')

# Must rebuild architecture first
fresh_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])
fresh_model.load_weights('/tmp/cnn_weights.weights.h5')
fresh_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
_, acc_w = fresh_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f'Weights-only format — Loaded acc: {acc_w*100:.2f}%')

---
## 1️⃣2️⃣ Interview Questions — Live Demonstrations

In [ ]:
# Q1: What is the difference between TF 1.x and TF 2.x?
import tensorflow as tf

print('TF 2.x — Eager Execution (default):')
print('Runs immediately, like Python/NumPy')

a = tf.constant(3.0)
b = tf.constant(4.0)
c = a * b
print(f'  a * b = {c.numpy()}  ← result available immediately\n')

print('TF 1.x style (static graph) — for comparison:')
print('  1. Build graph: a = tf.placeholder(); b = tf.placeholder(); c = a * b')
print('  2. Run in session: with tf.Session() as sess: sess.run(c, feed_dict={a:3, b:4})')
print('  3. Result not available until sess.run() called')
print()
print('TF 2.x benefits:')
print('  ✓ Easier debugging (print values anywhere)')
print('  ✓ Pythonic (if/for work naturally)')
print('  ✓ Use @tf.function for performance when needed')

In [ ]:
# Q2: What does tf.data.AUTOTUNE do?
import tensorflow as tf
import time

def slow_preprocess(x, y):
    # Simulate slow preprocessing
    tf.py_function(lambda: None, [], [])
    return x * 2.0, y

ds_slow = (
    tf.data.Dataset.from_tensor_slices((X_train_flat, y_train))
    .batch(256)
)

ds_fast = (
    tf.data.Dataset.from_tensor_slices((X_train_flat, y_train))
    .batch(256)
    .prefetch(tf.data.AUTOTUNE)   # Pre-fetches next batch while model trains
    .cache()                      # Cache dataset in memory after first epoch
)

print('AUTOTUNE explanation:')
print('  .prefetch(AUTOTUNE) — TF decides how many batches to pre-load')
print('  .map(fn, num_parallel_calls=AUTOTUNE) — TF decides parallelism')
print('  .cache() — stores dataset in RAM after first epoch')
print('  Result: CPU preprocessing overlaps with GPU training')

In [ ]:
# Q3: training=True vs training=False — why does it matter?
import tensorflow as tf
from tensorflow.keras import layers

# Dropout behaves differently in train vs inference
drop = layers.Dropout(0.5)
x    = tf.ones((1, 10))

print('Dropout training=True  (active):')
for _ in range(3):
    print('  ', drop(x, training=True).numpy())

print('\nDropout training=False (inactive, values scaled):')
for _ in range(3):
    print('  ', drop(x, training=False).numpy())

print()
print('BatchNormalization also differs:')
bn = layers.BatchNormalization()
batch_data = tf.random.normal((32, 10))

out_train = bn(batch_data, training=True)   # Uses batch mean/var, updates running stats
out_infer = bn(batch_data, training=False)  # Uses running stats (accumulated from training)
print(f'  Train mode mean: {out_train.numpy().mean():.4f} (close to 0)')
print(f'  Infer mode mean: {out_infer.numpy().mean():.4f}')
print('\nAlways pass training=True during .fit(), training=False during .evaluate()/.predict()')

In [ ]:
# Q4: from_logits=True vs from_logits=False
import tensorflow as tf
import numpy as np

logits  = np.array([[2.0, 1.0, 0.1]], dtype=np.float32)  # Raw model output
probs   = tf.nn.softmax(logits)                            # After softmax
y_true  = np.array([0])                                    # True class

loss_with_logits   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)(y_true, logits)
loss_with_probs    = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)(y_true, probs)

print(f'Logits          : {logits}')
print(f'Probs (softmax) : {probs.numpy().round(4)}')
print(f'Loss from_logits=True  : {loss_with_logits.numpy():.6f}')
print(f'Loss from_logits=False : {loss_with_probs.numpy():.6f}')
print(f'Are equal: {abs(loss_with_logits.numpy() - loss_with_probs.numpy()) < 1e-5}')
print()
print('from_logits=True is NUMERICALLY SAFER because TF combines softmax + crossentropy')
print('internally using the log-sum-exp trick, avoiding float overflow/underflow.')

---
## 📊 Final Summary — All 3 Models on MNIST

In [ ]:
mlp_loss, mlp_acc   = model.evaluate(X_test_flat, y_test, verbose=0)

print('\n' + '='*60)
print(f'{"Model":<20} {"Test Accuracy":>15} {"Params":>15}')
print('-'*60)
print(f'{"MLP (Dense)":<20} {mlp_acc*100:>14.2f}%  {model.count_params():>14,}')
print(f'{"CNN":<20} {cnn_acc*100:>14.2f}%  {cnn_model.count_params():>14,}')
print(f'{"LSTM (Bi)":<20} {lstm_acc*100:>14.2f}%  {lstm_model.count_params():>14,}')
print('='*60)
print()
print('Key TensorFlow concepts to remember:')
print('  ✓ tf.constant = immutable, tf.Variable = mutable (model weights)')
print('  ✓ GradientTape records ops to compute gradients')
print('  ✓ @tf.function compiles to fast graph (use inside training loop)')
print('  ✓ tf.data pipeline: .shuffle → .batch → .map → .cache → .prefetch')
print('  ✓ training=True/False controls Dropout and BatchNorm behavior')
print('  ✓ from_logits=True is numerically safer')

---
## ✅ Quick Reference Cheatsheet

```python
# ── Tensors ────────────────────────────────────────────────
t = tf.constant([1.0, 2.0])            # Immutable
v = tf.Variable(tf.zeros((3,3)))       # Mutable (model params)
v.assign(new_value)                    # Update variable

# ── Gradients ──────────────────────────────────────────────
with tf.GradientTape() as tape:
    loss = loss_fn(y_true, model(x, training=True))
grads = tape.gradient(loss, model.trainable_weights)
optimizer.apply_gradients(zip(grads, model.trainable_weights))

# ── Build, Compile, Train ──────────────────────────────────
model = keras.Sequential([...])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, validation_split=0.2)

# ── tf.data ────────────────────────────────────────────────
ds = tf.data.Dataset.from_tensor_slices((X, y))
ds = ds.shuffle(10000).batch(32).map(preprocess).prefetch(tf.data.AUTOTUNE)

# ── Compile to graph ────────────────────────────────────────
@tf.function
def train_step(x, y):
    ...

# ── Save/Load ──────────────────────────────────────────────
model.save('model.keras')
model = keras.models.load_model('model.keras')
```

---
*Last updated: May 2026*